# Gold — Clientes por perfil de renda da UF

Desenvolvido por: Ygor Moraes

Este notebook cria a Gold de enriquecimento dos clientes por perfil socioeconômico da UF.

Fontes:
- Silver `ecommerce_enderecos`
- Silver `ibge_renda_uf`

Regra:
- considerar apenas endereço principal;
- cruzar a UF do endereço com a renda per capita do IBGE;
- agregar clientes por estado e faixa/perfil de renda.

Destino ADLS:
- `gold/ecommerce_enderecos/clientes_perfil_renda_uf`

Destino SQL Server:
- `squad3.gold_ecommerce_enderecos_clientes_perfil_renda_uf`

In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
# Define imports, caminhos e parâmetros da Gold.

from pyspark.sql.functions import (
    col,
    count,
    countDistinct,
    current_timestamp,
    round as spark_round,
    desc,
    coalesce,
    lit,
    trim,
    upper,
    when,
    sum as spark_sum
)

SILVER_ENDERECOS_TABLE = "ecommerce_enderecos"
SILVER_IBGE_TABLE = "ibge_renda_uf"

SILVER_ENDERECOS_PATH = f"{SILVER_BASE_PATH}{SILVER_ENDERECOS_TABLE}"
SILVER_IBGE_PATH = f"{SILVER_BASE_PATH}{SILVER_IBGE_TABLE}"

GOLD_DOMAIN = "ecommerce_enderecos"
GOLD_KPI = "clientes_perfil_renda_uf"

GOLD_PATH = f"{GOLD_BASE_PATH}{GOLD_DOMAIN}/{GOLD_KPI}"

FINAL_TABLE_NAME = f"gold_{GOLD_DOMAIN}_{GOLD_KPI}"
FINAL_TABLE = f"{TARGET_SCHEMA}.{FINAL_TABLE_NAME}"

GOLD_WRITE_MODE = "overwrite"

ENDERECOS_REQUIRED_COLUMNS = [
    "id_endereco",
    "id_cliente",
    "estado",
    "cidade",
    "cep",
    "is_principal"
]

IBGE_REQUIRED_COLUMNS = [
    "uf",
    "nome_uf",
    "renda_media_per_capita",
    "ano_referencia",
    "fonte"
]

GOLD_KEY_COLUMNS = [
    "estado",
]

adls_options = get_adls_options()

print("Parâmetros definidos com sucesso.")
print("SILVER_ENDERECOS_PATH:", SILVER_ENDERECOS_PATH)
print("SILVER_IBGE_PATH:", SILVER_IBGE_PATH)
print("GOLD_PATH:", GOLD_PATH)
print("FINAL_TABLE:", FINAL_TABLE)

In [0]:
# Lê as Silvers de endereços e renda do IBGE.

df_enderecos = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(SILVER_ENDERECOS_PATH)
)

df_ibge = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(SILVER_IBGE_PATH)
)

validate_required_columns(df_enderecos, ENDERECOS_REQUIRED_COLUMNS)

validate_required_columns(df_ibge, IBGE_REQUIRED_COLUMNS)

print("Silver de endereços lida com sucesso.")
print(f"Total de endereços: {df_enderecos.count()}")
print("Colunas endereços:", df_enderecos.columns)

df_enderecos.printSchema()

display(df_enderecos.limit(10))

print("Silver de renda IBGE lida com sucesso.")
print(f"Total de registros IBGE: {df_ibge.count()}")
print("Colunas IBGE:", df_ibge.columns)

df_ibge.printSchema()

display(df_ibge.limit(10))

In [0]:
# Prepara as bases para o join por UF.

df_enderecos_principais = (
    df_enderecos
    .filter(col("is_principal") == True)
    .select(
        "id_cliente",
        "estado",
        "cidade"
    )
    .withColumn("estado", upper(trim(col("estado"))))
    .withColumn("cidade", trim(col("cidade")))
)

df_ibge_uf = (
    df_ibge
    .select(
        upper(trim(col("uf"))).alias("uf"),
        "nome_uf",
        "renda_media_per_capita",
        "ano_referencia",
        "fonte"
    )
)

print("Bases preparadas para o join.")
print(f"Clientes com endereço principal: {df_enderecos_principais.select('id_cliente').distinct().count()}")
print(f"UFs na base de endereços: {df_enderecos_principais.select('estado').distinct().count()}")
print(f"UFs na base IBGE: {df_ibge_uf.select('uf').distinct().count()}")

display(
    df_enderecos_principais
    .groupBy("estado")
    .agg(countDistinct("id_cliente").alias("qtd_clientes"))
    .orderBy(desc("qtd_clientes"))
)

In [0]:
# Cruza clientes com renda média da UF.

df_enderecos_ibge = (
    df_enderecos_principais
    .join(
        df_ibge_uf,
        df_enderecos_principais["estado"] == df_ibge_uf["uf"],
        "left"
    )
    .drop("uf")
    .withColumn(
        "fl_match_ibge",
        when(col("renda_media_per_capita").isNotNull(), 1).otherwise(0)
    )
)

total_clientes_principais = (
    df_enderecos_principais
    .select("id_cliente")
    .distinct()
    .count()
)

clientes_sem_match_ibge = (
    df_enderecos_ibge
    .filter(col("fl_match_ibge") == 0)
    .select("id_cliente")
    .distinct()
    .count()
)

print("Join com IBGE realizado.")
print(f"Total clientes principais: {total_clientes_principais}")
print(f"Clientes sem match com IBGE: {clientes_sem_match_ibge}")

display(df_enderecos_ibge.limit(10))

In [0]:
# Agrega clientes por UF com renda média per capita do IBGE.

df_gold = (
    df_enderecos_ibge
    .groupBy(
        "estado",
        "nome_uf",
        "renda_media_per_capita",
        "ano_referencia",
        "fonte"
    )
    .agg(
        countDistinct("id_cliente").alias("qtd_clientes"),
        spark_sum("fl_match_ibge").alias("qtd_clientes_com_match_ibge"),
        countDistinct("cidade").alias("qtd_cidades")
    )
    .withColumn(
        "qtd_clientes_sem_match_ibge",
        col("qtd_clientes") - col("qtd_clientes_com_match_ibge")
    )
    .withColumn(
        "percentual_clientes",
        spark_round((col("qtd_clientes") / lit(total_clientes_principais)) * 100, 2)
    )
    .withColumn(
        "percentual_match_ibge",
        spark_round((col("qtd_clientes_com_match_ibge") / col("qtd_clientes")) * 100, 2)
    )
    .withColumn("gold_processed_at", current_timestamp())
    .orderBy("estado")
)

print("Gold agregada com sucesso.")
print(f"Total de linhas Gold: {df_gold.count()}")

display(df_gold)

In [0]:
# Valida chave da Gold e consistência da contagem de clientes.

total_linhas_gold = df_gold.count()

estados_distintos_gold = (
    df_gold
    .select(*GOLD_KEY_COLUMNS)
    .distinct()
    .count()
)

estados_duplicados = total_linhas_gold - estados_distintos_gold

validacao_gold = (
    df_gold
    .agg(
        spark_sum("qtd_clientes").alias("total_clientes_gold"),
        spark_sum("qtd_clientes_com_match_ibge").alias("total_clientes_com_match_ibge"),
        spark_sum("qtd_clientes_sem_match_ibge").alias("total_clientes_sem_match_ibge")
    )
    .collect()[0]
)

print(f"Total de linhas Gold: {total_linhas_gold}")
print(f"Estados duplicados: {estados_duplicados}")
print(f"Total clientes principais: {total_clientes_principais}")
print(f"Total clientes Gold: {validacao_gold['total_clientes_gold']}")
print(f"Clientes com match IBGE: {validacao_gold['total_clientes_com_match_ibge']}")
print(f"Clientes sem match IBGE: {validacao_gold['total_clientes_sem_match_ibge']}")

if estados_duplicados != 0:
    raise ValueError("Validação falhou: existem estados duplicados na Gold.")

if validacao_gold["total_clientes_gold"] != total_clientes_principais:
    raise ValueError("Validação falhou: total de clientes da Gold diferente da base principal.")

if (
    validacao_gold["total_clientes_com_match_ibge"] +
    validacao_gold["total_clientes_sem_match_ibge"]
    != validacao_gold["total_clientes_gold"]
):
    raise ValueError("Validação falhou: clientes com + sem match IBGE não fecha com total.")

print("Validação OK: Gold sem duplicidade e com clientes consistentes.")

In [0]:
# Grava a Gold em Delta no ADLS.

(
    df_gold
    .write
    .format("delta")
    .options(**adls_options)
    .option("overwriteSchema", "true")
    .mode(GOLD_WRITE_MODE)
    .save(GOLD_PATH)
)

print("Gold gravada com sucesso no ADLS.")
print("Caminho:", GOLD_PATH)

In [0]:
# Lê a Gold gravada no ADLS.

df_gold_gravada = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(GOLD_PATH)
)

print("Gold lida com sucesso do ADLS.")
print(f"Total de linhas gravadas: {df_gold_gravada.count()}")

display(df_gold_gravada.orderBy("estado"))

In [0]:
# Valida a Gold gravada no ADLS.

total_linhas_gold_gravada = df_gold_gravada.count()

estados_distintos_gold_gravada = (
    df_gold_gravada
    .select(*GOLD_KEY_COLUMNS)
    .distinct()
    .count()
)

estados_duplicados_gravada = total_linhas_gold_gravada - estados_distintos_gold_gravada

validacao_gold_gravada = (
    df_gold_gravada
    .agg(
        spark_sum("qtd_clientes").alias("total_clientes_gold"),
        spark_sum("qtd_clientes_com_match_ibge").alias("total_clientes_com_match_ibge"),
        spark_sum("qtd_clientes_sem_match_ibge").alias("total_clientes_sem_match_ibge")
    )
    .collect()[0]
)

print(f"Total de linhas Gold gravada: {total_linhas_gold_gravada}")
print(f"Estados duplicados Gold gravada: {estados_duplicados_gravada}")
print(f"Total clientes principais: {total_clientes_principais}")
print(f"Total clientes Gold gravada: {validacao_gold_gravada['total_clientes_gold']}")
print(f"Clientes com match IBGE: {validacao_gold_gravada['total_clientes_com_match_ibge']}")
print(f"Clientes sem match IBGE: {validacao_gold_gravada['total_clientes_sem_match_ibge']}")

if estados_duplicados_gravada != 0:
    raise ValueError("Validação falhou: existem estados duplicados na Gold gravada.")

if validacao_gold_gravada["total_clientes_gold"] != total_clientes_principais:
    raise ValueError("Validação falhou: total de clientes da Gold gravada diferente da base principal.")

if (
    validacao_gold_gravada["total_clientes_com_match_ibge"] +
    validacao_gold_gravada["total_clientes_sem_match_ibge"]
    != validacao_gold_gravada["total_clientes_gold"]
):
    raise ValueError("Validação falhou: clientes com + sem match IBGE não fecha com total.")

print("Validação OK: Gold gravada corretamente no ADLS.")

In [0]:
# Prepara a Gold para gravação no SQL Server.

df_gold_sql = (
    df_gold_gravada
    .select(
        "estado",
        "nome_uf",
        "renda_media_per_capita",
        "ano_referencia",
        "fonte",
        "qtd_clientes",
        "qtd_clientes_com_match_ibge",
        "qtd_clientes_sem_match_ibge",
        "qtd_cidades",
        "percentual_clientes",
        "percentual_match_ibge",
        "gold_processed_at"
    )
)

print("Gold preparada para SQL Server.")
print(f"Total de linhas: {df_gold_sql.count()}")
print("Tabela destino:", FINAL_TABLE)

display(df_gold_sql.orderBy("estado"))

In [0]:
# Grava a Gold diretamente no SQL Server.

write_sql_table(
    df=df_gold_sql,
    sql_host=SQL_HOST,
    sql_database=SQL_DATABASE,
    sql_username=SQL_USERNAME,
    sql_password=SQL_PASSWORD,
    table_name=FINAL_TABLE,
    mode="overwrite",
    sql_port=SQL_PORT
)

print("Gold gravada com sucesso no SQL Server.")
print("Tabela:", FINAL_TABLE)

In [0]:
# Lê a tabela final no SQL Server para validação.

df_final_sql = read_sql_table(
    spark=spark,
    sql_host=SQL_HOST,
    sql_database=SQL_DATABASE,
    sql_username=SQL_USERNAME,
    sql_password=SQL_PASSWORD,
    table_name=FINAL_TABLE,
    sql_port=SQL_PORT
)

print("Tabela final lida com sucesso do SQL Server.")
print(f"Total de linhas SQL Server: {df_final_sql.count()}")

display(df_final_sql.orderBy("estado"))

In [0]:
# Valida a tabela final no SQL Server.

total_linhas_sql = df_final_sql.count()

estados_distintos_sql = (
    df_final_sql
    .select(*GOLD_KEY_COLUMNS)
    .distinct()
    .count()
)

estados_duplicados_sql = total_linhas_sql - estados_distintos_sql

validacao_sql = (
    df_final_sql
    .agg(
        spark_sum("qtd_clientes").alias("total_clientes_sql"),
        spark_sum("qtd_clientes_com_match_ibge").alias("total_clientes_com_match_ibge"),
        spark_sum("qtd_clientes_sem_match_ibge").alias("total_clientes_sem_match_ibge")
    )
    .collect()[0]
)

print(f"Total de linhas Gold ADLS: {total_linhas_gold_gravada}")
print(f"Total de linhas SQL Server: {total_linhas_sql}")
print(f"Estados duplicados SQL Server: {estados_duplicados_sql}")
print(f"Total clientes Gold ADLS: {validacao_gold_gravada['total_clientes_gold']}")
print(f"Total clientes SQL Server: {validacao_sql['total_clientes_sql']}")
print(f"Clientes com match IBGE SQL Server: {validacao_sql['total_clientes_com_match_ibge']}")
print(f"Clientes sem match IBGE SQL Server: {validacao_sql['total_clientes_sem_match_ibge']}")

if total_linhas_sql != total_linhas_gold_gravada:
    raise ValueError("Validação falhou: total de linhas no SQL Server diferente da Gold no ADLS.")

if estados_duplicados_sql != 0:
    raise ValueError("Validação falhou: existem estados duplicados no SQL Server.")

if validacao_sql["total_clientes_sql"] != validacao_gold_gravada["total_clientes_gold"]:
    raise ValueError("Validação falhou: total de clientes no SQL Server diferente da Gold no ADLS.")

if (
    validacao_sql["total_clientes_com_match_ibge"] +
    validacao_sql["total_clientes_sem_match_ibge"]
    != validacao_sql["total_clientes_sql"]
):
    raise ValueError("Validação falhou: clientes com + sem match IBGE não fecha com total no SQL Server.")

print("Validação OK: tabela final SQL Server gravada corretamente.")